# 01. Exploratory Data Analysis: Ujjain Onion Mandi Prices
**Project:** Ujjain Onion Price Prediction System  
**Source:** Directorate of Marketing & Inspection (DMI), AGMARKNET & data.gov.in  
**Market:** Ujjain APMC (F&V), Madhya Pradesh  
**Commodity:** Onion (Wholesale)

This notebook conducts end-to-end exploratory analysis on authentic government mandi records.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set project root
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 1. Data Ingestion & Authenticity Check

In [ ]:
data_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'ujjain_onion_clean.csv')
if not os.path.exists(data_path):
    data_path = 'data/processed/ujjain_onion_clean.csv'

df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Total verified Ujjain Onion records: {len(df)}")
print(f"Date span: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Missing values in dataset: {df.isnull().sum().sum()}")
df.head()

## 2. Summary Statistics

In [ ]:
stats = df[['min_price', 'max_price', 'modal_price']].describe()
print("Price Summary Statistics (in Rs. per Quintal):")
stats

## 3. Daily Price Trends & Min-Max Spread
Visualizing the historical trajectory of wholesale onion modal price in Ujjain APMC alongside the daily minimum and maximum price envelopes.

In [ ]:
plt.figure(figsize=(14, 6))
plt.fill_between(df['date'], df['min_price'], df['max_price'], color='orange', alpha=0.2, label='Min-Max Price Spread')
plt.plot(df['date'], df['modal_price'], color='#d9534f', linewidth=2, label='Modal Price (Rs./Quintal)')
plt.title('Ujjain APMC: Wholesale Onion Price History (2020)', fontsize=14, fontweight='bold')
plt.xlabel('Auction Date')
plt.ylabel('Price (Rs. / Quintal)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 4. Rolling Moving Averages (7-Day, 14-Day, 30-Day)
Smoothing high-frequency auction fluctuations using trailing moving averages.

In [ ]:
df['ma_7'] = df['modal_price'].rolling(window=7, min_periods=1).mean()
df['ma_14'] = df['modal_price'].rolling(window=14, min_periods=1).mean()
df['ma_30'] = df['modal_price'].rolling(window=30, min_periods=1).mean()

plt.figure(figsize=(14, 6))
plt.plot(df['date'], df['modal_price'], color='lightgray', linewidth=1.5, label='Actual Modal Price')
plt.plot(df['date'], df['ma_7'], color='#0275d8', linewidth=2, label='7-Period Moving Average')
plt.plot(df['date'], df['ma_14'], color='#5cb85c', linewidth=2, label='14-Period Moving Average')
plt.plot(df['date'], df['ma_30'], color='#f0ad4e', linewidth=2.5, linestyle='--', label='30-Period Moving Average')
plt.title('Ujjain Onion Price with Trailing Moving Averages', fontsize=14, fontweight='bold')
plt.xlabel('Auction Date')
plt.ylabel('Price (Rs. / Quintal)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Price Distribution & Spread Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
sns.histplot(df['modal_price'], kde=True, ax=axes[0], color='#d9534f', bins=20)
axes[0].set_title('Modal Price Distribution (Histogram & KDE)', fontweight='bold')
axes[0].set_xlabel('Modal Price (Rs. / Quintal)')
axes[0].set_ylabel('Trading Frequency')

# Boxplot
sns.boxplot(x=df['modal_price'], ax=axes[1], color='#f0ad4e')
axes[1].set_title('Modal Price Dispersion (Boxplot)', fontweight='bold')
axes[1].set_xlabel('Modal Price (Rs. / Quintal)')

plt.tight_layout()
plt.show()

## 6. Daily & Weekly Price Changes & Volatility

In [ ]:
df['daily_change'] = df['modal_price'].diff()
df['weekly_change'] = df['modal_price'].diff(7)
df['percentage_change'] = df['modal_price'].pct_change() * 100.0

plt.figure(figsize=(14, 5))
plt.bar(df['date'], df['daily_change'], color=np.where(df['daily_change'] >= 0, '#5cb85c', '#d9534f'), width=1.5)
plt.title('Daily Price Changes (Rs. / Quintal Delta)', fontsize=14, fontweight='bold')
plt.xlabel('Auction Date')
plt.ylabel('Delta vs Previous Trading Day (Rs./Q)')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Monthly Trends & Seasonal Aggregation

In [ ]:
df['month_name'] = df['date'].dt.strftime('%b %Y')
monthly = df.groupby('month_name', sort=False)['modal_price'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
print("Monthly Price Aggregations:")
display(monthly) if 'display' in locals() else print(monthly)

plt.figure(figsize=(12, 5))
plt.bar(monthly['month_name'], monthly['mean'], yerr=monthly['std'], capsize=5, color='#0275d8', alpha=0.8)
plt.title('Monthly Average Modal Price (Ujjain APMC)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Mean Modal Price +/- 1 Std Dev (Rs./Q)')
plt.tight_layout()
plt.show()

## 8. Missing Dates & Pandemic Lockdown Analysis
Analyzing the calendar gaps, non-trading weekend closures, and the nationwide agricultural market closures in March-May 2020.

In [ ]:
df['date_diff'] = df['date'].diff().dt.days
gaps = df[df['date_diff'] > 2][['date', 'date_diff', 'modal_price']]
print("Significant Non-Trading Gaps (>2 Days):")
print(gaps)

lockdown_gap = gaps[gaps['date_diff'] > 30]
if not lockdown_gap.empty:
    print("\nVerified COVID-19 Lockdown Market Suspension:")
    print(f"Market reopened on {lockdown_gap['date'].iloc[0].date()} after a gap of {int(lockdown_gap['date_diff'].iloc[0])} calendar days.")